## Setup & Environment

Install required libraries, check GPU, and mount Google Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

train = pd.read_csv("/content/drive/MyDrive/Project2_preprocessing_mal_en/clean_train.csv")
dev   = pd.read_csv("/content/drive/MyDrive/Project2_preprocessing_mal_en/clean_dev.csv")
test  = pd.read_csv("/content/drive/MyDrive/Project2_preprocessing_mal_en/clean_test.csv")

print("Train shape:", train.shape)
print("Dev shape:",   dev.shape)
print("Test shape:",  test.shape)

Train shape: (4840, 5)
Dev shape: (540, 4)
Test shape: (1347, 4)


In [3]:
# Keep only Positive and Negative
train = train[train["category"].isin(["Positive", "Negative"])]
dev   = dev[dev["category"].isin(["Positive", "Negative"])]
test  = test[test["category"].isin(["Positive", "Negative"])]

print("Filtered Train shape:", train.shape)
print("Filtered Dev shape:",   dev.shape)
print("Filtered Test shape:",  test.shape)

Filtered Train shape: (2566, 5)
Filtered Dev shape: (275, 4)
Filtered Test shape: (702, 4)


## Convert to mT5 Format

In [4]:
def convert_to_finetune_format(df):
    formatted = []
    for _, row in df.iterrows():
        formatted.append({
            "input_text":  "sentiment: " + str(row["text"]),
            "target_text": row["category"].lower().strip()
        })
    return formatted

train_data = convert_to_finetune_format(train)
dev_data   = convert_to_finetune_format(dev)
test_data  = convert_to_finetune_format(test)

print(f"Prepared {len(train_data)} train, {len(dev_data)} dev, {len(test_data)} test examples")

Prepared 2566 train, 275 dev, 702 test examples


## Model + Tokenizer

In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print("Model loaded:", MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded: google/mt5-small


## Tokenization

In [6]:
def tokenize_example(example):
    model_input = tokenizer(
        example["input_text"],
        max_length=64,
        padding="max_length",
        truncation=True
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            example["target_text"],
            max_length=8,
            padding="max_length",
            truncation=True
        )

    label_ids = [
        (token if token != tokenizer.pad_token_id else -100)
        for token in labels["input_ids"]
    ]

    model_input["labels"] = label_ids
    return model_input

## Create HuggingFace Datasets

In [7]:
from datasets import Dataset

# Redefine tokenize_example to fix the AttributeError
def tokenize_example(example):
    model_input = tokenizer(
        example["input_text"],
        max_length=64,
        padding="max_length",
        truncation=True
    )

    # Removed 'with tokenizer.as_target_tokenizer():' as it's deprecated
    labels = tokenizer(
        example["target_text"],
        max_length=8,
        padding="max_length",
        truncation=True
    )

    label_ids = [
        (token if token != tokenizer.pad_token_id else -100)
        for token in labels["input_ids"]
    ]

    model_input["labels"] = label_ids
    return model_input

train_dataset = Dataset.from_list(train_data)
dev_dataset   = Dataset.from_list(dev_data)
test_dataset  = Dataset.from_list(test_data)

tokenized_train = train_dataset.map(tokenize_example)
tokenized_dev   = dev_dataset.map(tokenize_example)
tokenized_test  = test_dataset.map(tokenize_example)

print("Tokenization done.")
print("Train:", len(tokenized_train), "Dev:", len(tokenized_dev), "Test:", len(tokenized_test))

Map:   0%|          | 0/2566 [00:00<?, ? examples/s]

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Map:   0%|          | 0/702 [00:00<?, ? examples/s]

Tokenization done.
Train: 2566 Dev: 275 Test: 702


## Normalize Labels & Compute Metrics

In [8]:
import re
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

ALLOWED_LABELS = {"positive", "negative"}

def normalize_label(text):
    if text is None:
        return "negative"
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    text = text.split()[0] if text.split() else ""
    return text if text in ALLOWED_LABELS else "negative"

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # preds may be logits (3D) or token ids (2D)
    if isinstance(preds, tuple):
        preds = preds[0]
    if preds.ndim == 3:
        preds = preds.argmax(-1)

    # Replace -100 in labels with pad token id before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    norm_preds  = [normalize_label(p) for p in decoded_preds]
    norm_labels = [normalize_label(l) for l in decoded_labels]

    acc = accuracy_score(norm_labels, norm_preds)
    f1  = f1_score(norm_labels, norm_preds, average="weighted", zero_division=0)

    print(f"\n--- Dev Metrics --- Accuracy: {acc:.4f} | F1 (weighted): {f1:.4f}")
    return {"accuracy": acc, "f1_weighted": f1}

print("compute_metrics defined.")

compute_metrics defined.


## Training Arguments

Key fixes:
- `eval_strategy='epoch'` so evaluation runs after each epoch
- `predict_with_generate=True` so the seq2seq model generates text for evaluation
- `load_best_model_at_end=True` to keep the best checkpoint
- Added `generation_max_length` to match label length

In [9]:
from transformers import Seq2SeqTrainingArguments
import torch

training_args = Seq2SeqTrainingArguments(
    output_dir="./outputs",

    # ---- FIXED: enable evaluation ----
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    save_total_limit=2,

    # ---- FIXED: use generate for seq2seq evaluation ----
    predict_with_generate=True,
    generation_max_length=8,

    learning_rate=3e-4,          # higher LR helps small mT5 converge faster
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
print("Training args configured.")

Training args configured.


## Trainer

In [10]:
from transformers import Seq2SeqTrainer, DataCollatorForSeq2Seq

# DataCollator handles dynamic padding properly
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,        # FIXED: pass dev set
    compute_metrics=compute_metrics,   # FIXED: pass metrics function
    data_collator=data_collator,
)
print("Trainer ready.")

Trainer ready.


## Train

In [11]:
train_result = trainer.train()
print("\n=== Training complete ===")
print(f"  Train loss:    {train_result.training_loss:.4f}")
print(f"  Train runtime: {train_result.metrics.get('train_runtime', 0):.1f}s")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,2.970045,1.130294,0.814545,0.731295
2,1.007301,0.527544,0.814545,0.731295
3,0.655194,0.351936,0.814545,0.731295



--- Dev Metrics --- Accuracy: 0.8145 | F1 (weighted): 0.7313


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



--- Dev Metrics --- Accuracy: 0.8145 | F1 (weighted): 0.7313


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].



=== Training complete ===
  Train loss:    4.7199
  Train runtime: 5309.8s


## Evaluate on Dev Set

In [12]:
dev_metrics = trainer.evaluate(eval_dataset=tokenized_dev)
print("\n=== Dev Set Results ===")
for k, v in dev_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



--- Dev Metrics --- Accuracy: 0.8145 | F1 (weighted): 0.7313

=== Dev Set Results ===
  eval_loss: 1.1303
  eval_accuracy: 0.8145
  eval_f1_weighted: 0.7313
  eval_runtime: 69.9196
  eval_samples_per_second: 3.9330
  eval_steps_per_second: 0.2570
  epoch: 3.0000


## Evaluate on Test Set

In [13]:
# Tokenize test set and run prediction
test_preds = trainer.predict(tokenized_test)

# Decode predictions
raw_preds = test_preds.predictions
if isinstance(raw_preds, tuple):
    raw_preds = raw_preds[0]
if raw_preds.ndim == 3:
    raw_preds = raw_preds.argmax(-1)

decoded_test_preds  = tokenizer.batch_decode(raw_preds, skip_special_tokens=True)
test_labels_arr     = np.where(test_preds.label_ids != -100,
                                test_preds.label_ids,
                                tokenizer.pad_token_id)
decoded_test_labels = tokenizer.batch_decode(test_labels_arr, skip_special_tokens=True)

norm_test_preds  = [normalize_label(p) for p in decoded_test_preds]
norm_test_labels = [normalize_label(l) for l in decoded_test_labels]

test_acc = accuracy_score(norm_test_labels, norm_test_preds)
test_f1  = f1_score(norm_test_labels, norm_test_preds, average="weighted", zero_division=0)

print("\n=== Test Set Results ===")
print(f"  Accuracy:    {test_acc:.4f}")
print(f"  F1 (weighted): {test_f1:.4f}")

# Show a sample of predictions
print("\n--- Sample Predictions (first 10) ---")
for i in range(min(10, len(norm_test_preds))):
    print(f"  Pred: {norm_test_preds[i]:<10}  Gold: {norm_test_labels[i]}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



--- Dev Metrics --- Accuracy: 0.8034 | F1 (weighted): 0.7158

=== Test Set Results ===
  Accuracy:    0.8034
  F1 (weighted): 0.7158

--- Sample Predictions (first 10) ---
  Pred: positive    Gold: negative
  Pred: positive    Gold: positive
  Pred: positive    Gold: negative
  Pred: positive    Gold: negative
  Pred: positive    Gold: positive
  Pred: positive    Gold: negative
  Pred: positive    Gold: positive
  Pred: positive    Gold: positive
  Pred: positive    Gold: negative
  Pred: positive    Gold: positive


## Save Model & Training History

In [16]:
import os

# Save locally in Colab first, then copy to Drive
save_path = "/content/mt5_finetuned"
os.makedirs(save_path, exist_ok=True)
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model saved locally to: {save_path}")

# Now copy to Drive
drive_path = "/content/drive/MyDrive/mt5_finetuned"
os.system(f"cp -r {save_path} '{drive_path}'")
print(f"Copied to Drive: {drive_path}")

# Show training log
print("\n--- Training Log History ---")
for entry in trainer.state.log_history:
    relevant = {k: round(v, 4) if isinstance(v, float) else v
                for k, v in entry.items()
                if k in ("epoch", "loss", "eval_accuracy", "eval_f1_weighted",
                         "eval_loss", "train_loss")}
    if relevant:
        print(relevant)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved locally to: /content/mt5_finetuned
Copied to Drive: /content/drive/MyDrive/mt5_finetuned

--- Training Log History ---
{'loss': 24.8214, 'epoch': 0.3106}
{'loss': 10.7833, 'epoch': 0.6211}
{'loss': 2.97, 'epoch': 0.9317}
{'eval_loss': 1.1303, 'eval_accuracy': 0.8145, 'eval_f1_weighted': 0.7313, 'epoch': 1.0}
{'loss': 1.7716, 'epoch': 1.2422}
{'loss': 1.5374, 'epoch': 1.5528}
{'loss': 1.0073, 'epoch': 1.8634}
{'eval_loss': 0.5275, 'eval_accuracy': 0.8145, 'eval_f1_weighted': 0.7313, 'epoch': 2.0}
{'loss': 0.8848, 'epoch': 2.1739}
{'loss': 0.7829, 'epoch': 2.4845}
{'loss': 0.6552, 'epoch': 2.795}
{'eval_loss': 0.3519, 'eval_accuracy': 0.8145, 'eval_f1_weighted': 0.7313, 'epoch': 3.0}
{'train_loss': 4.7199, 'epoch': 3.0}
{'eval_loss': 1.1303, 'eval_accuracy': 0.8145, 'eval_f1_weighted': 0.7313, 'epoch': 3.0}
